# Day 1 — ILT 4: Intro to Databricks for Data Engineers + PySpark & Spark SQL
**Duration:** 60 min ILT + 60 min Hands-on &nbsp;|&nbsp; **Level:** Beginner &nbsp;|&nbsp; **Tags:** databricks, pyspark, spark-sql, dataframe, ADLS

**Time:** 2:00 PM – 4:00 PM  
**Hands-on at 3:00 PM:** Data wrangling using PySpark & Spark SQL + Work with ADLS

---

## Learning Objectives

By the end of this session, students will be able to:

1. Explain what Databricks is and identify its key workspace components
2. Describe how a Spark cluster works — Driver coordinates, Workers process in parallel
3. Read CSV files from ADLS into a PySpark DataFrame
4. Apply essential PySpark transformations: `select`, `filter`, `withColumn`, `groupBy`, `join`
5. Write DataFrames to Delta tables in Bronze using `overwrite` and `append`
6. Query Delta tables using Spark SQL (`%sql` and `spark.sql()`)

---

### What we will cover today
1. What is Databricks and why Data Engineers use it
2. Key parts of the Databricks workspace + cluster architecture
3. PySpark DataFrame API — reading, exploring, and transforming data
4. Writing to Delta Lake — landing data in Bronze
5. Spark SQL — querying Delta tables with SQL

> **Instructor note:** 15 min on Databricks overview (show UI live), 25 min on PySpark DataFrame API with code, 20 min on Spark SQL. Hands-on at 3:00 PM.

## Section 1 — What is Databricks?

Databricks is a **cloud platform** built on top of **Apache Spark**.  
Think of it as a supercharged Jupyter notebook that can process millions of rows using many computers at the same time.

### Why do Data Engineers use Databricks?

| Problem | How Databricks Solves It |
|---------|-------------------------|
| Data is too big for one laptop | Uses many machines (a cluster) to share the work |
| Need to store data in a smart format | Uses Delta Lake (not just CSV files) |
| Need to schedule pipelines | Has built-in Workflows and Jobs |
| Need to manage code versions | Connects to GitHub via Databricks Repos |
| Need to govern who sees what data | Unity Catalog for permissions and lineage |

### Databricks vs Traditional Tools

```
Old way:  Laptop → Python script → CSV file on disk
New way:  Databricks cluster → PySpark → Delta Lake on Azure cloud
```

> The code looks almost the same as pandas, but it runs on 10 machines instead of 1.

## Section 2 — Key Parts of the Databricks Workspace

When you open Databricks, you see these sections on the left menu:

| Section | What it is | What we use it for |
|---------|-----------|--------------------|
| **Workspace** | Your notebook files | Write and run PySpark code |
| **Repos** | Connected to GitHub | Version control for notebooks |
| **Clusters** | The computing power | Must be running before you run code |
| **Jobs/Workflows** | Scheduled pipelines | Run notebooks automatically at a set time |
| **Data** | Browse tables and files | See Delta tables you have created |
| **SQL Editor** | SQL query tool | Query tables like a database |

### How a Cluster Works — Very Simple

```
  YOUR NOTEBOOK
       |
       v
   DRIVER NODE  ← This is the "brain" — it coordinates the work
   /    |    \
Worker Worker Worker  ← These are the "hands" — they process the data in parallel
```

- When you run a cell, the **Driver** reads your code and splits the data
- **Worker nodes** each process their part of the data
- Result comes back to the Driver and you see the output

> **For GlobalMart:** Our 6 ADLS CSV/JSON source files are small, so we only need a small cluster (the other 2 tables, `orders`/`order_items`, come from Postgres via Lakeflow Connect, not ADLS files).  
> In real companies, the data could be billions of rows — same code, bigger cluster.

## Section 3 — PySpark DataFrame API

PySpark is the Python API for Apache Spark. The core object is the **DataFrame** — a distributed table of rows and columns, similar to pandas but designed to run across a cluster of machines.

### Lazy Evaluation

PySpark is **lazy** — it builds a plan but does NOT execute until you call an **action**.

```
df = spark.read.csv(path)         ← No data moved — just a plan
df2 = df.filter(df.status == "pending")  ← Still just a plan
df2.show(5)                       ← THIS triggers execution — cluster processes data
```

This means you can chain 10 transformations and Spark will optimise them into one efficient job before running anything.

### Key Operations at a Glance

| Category | Method | What it does |
|----------|--------|--------------|
| **Read** | `spark.read.csv(path)` | Load CSV from ADLS |
| **Read** | `spark.read.format("delta").load(path)` | Load Delta table from path |
| **Explore** | `.printSchema()` | Column names and types |
| **Explore** | `.show(n)` / `.display()` | Show rows (display = interactive in Databricks) |
| **Explore** | `.count()` | Total row count — triggers execution |
| **Transform** | `.select("col1", "col2")` | Keep specific columns |
| **Transform** | `.filter(df.col == "val")` | Keep rows matching condition |
| **Transform** | `.withColumn("new", expr)` | Add or overwrite a column |
| **Transform** | `.dropna(subset=["col"])` | Remove rows with null in column |
| **Transform** | `.distinct()` | Remove duplicate rows |
| **Transform** | `.groupBy("col").agg(...)` | Aggregate rows by group |
| **Transform** | `.join(other, "key", "left")` | Join two DataFrames |
| **Write** | `.write.format("delta").mode("overwrite").save(path)` | Write Delta, full replace |
| **Write** | `.write.format("delta").mode("append").save(path)` | Write Delta, add rows |

> pandas runs on one machine. PySpark runs on a cluster. Same shape of API — very different scale.

## Setup — Connect to ADLS Gen2

Run this cell first. Every cell below depends on the variables set here.

In [ ]:
# ─── ADLS Connection Setup ───────────────────────────────────────────────────
# HOW TO GET THESE VALUES MANUALLY (Azure Portal → portal.azure.com):
#   storage_account_name → Storage Accounts → your own account → shown at
#                           the top of the Overview page
#   storage_account_key  → same account → left menu "Security + Networking"
#                           → "Access keys" → click "Show" next to key1 →
#                           Copy (an 88-character secret)
#   container_name        → same account → left menu "Data storage" →
#                           "Containers" — your own container from Day 1
# ⚠️ Never commit a real key to git or leave it in a saved/shared notebook.
#
# Note: this notebook writes its demo Delta tables to a scratch path under
# raw-data/, purely to practice the read/write/SQL mechanics. It is NOT
# where GlobalMart's real Bronze lives — that's a Unity Catalog schema
# (<your-catalog>.bronze.*), built properly starting Day 4.

storage_account_name = "YOUR_STORAGE_ACCOUNT_NAME"
container_name       = "YOUR_CONTAINER_NAME"
raw_folder           = "raw-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← Replace this!

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

raw_path        = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{raw_folder}"
delta_demo_path = f"{raw_path}/_delta_demo"   # scratch path for THIS notebook's demo only

print("Connected to ADLS!")
print(f"Raw path       : {raw_path}")
print(f"Delta demo path: {delta_demo_path}")

## Section 4 — Reading Data from ADLS

The first step in every pipeline: read the raw files from `raw-data/` in ADLS and load them into a PySpark DataFrame.

```
ADLS  raw-data/customers/customers.csv  →  spark.read.csv(...)  →  PySpark DataFrame
```

### Options you will always use

| Option | Value | Meaning |
|--------|-------|---------|
| `header` | `"true"` | First row is column names |
| `inferSchema` | `"true"` | Detect column types automatically |

> In production, always define the schema explicitly instead of inferring — it is faster and safer. For learning, `inferSchema=True` is fine.

In [ ]:
# ─── Read a CSV from ADLS into a PySpark DataFrame ───────────────────────────

# Read payment_methods — a small lookup table
payment_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{raw_path}/payment_methods/payment_methods.csv")
)

print("payment_methods — schema:")
payment_df.printSchema()

print("payment_methods — all rows:")
payment_df.show()
print(f"Total rows: {payment_df.count()}")

In [ ]:
# ─── Read customers.csv — explore with printSchema, show, count ──────────────

customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{raw_path}/customers/customers.csv")
)

print(f"Total rows: {customers_df.count()}")
print()
print("Schema — column names and data types:")
customers_df.printSchema()

print("First 5 rows:")
customers_df.show(5, truncate=False)

In [ ]:
# ─── PySpark transformations: select, filter, withColumn ─────────────────────
from pyspark.sql.functions import col, current_timestamp

# select — keep only specific columns
slim_df = customers_df.select("customer_id", "first_name", "last_name", "email", "city")
print("After select — 5 columns:")
slim_df.show(3)

# filter — keep rows matching a condition (pick a city from your actual data)
city_sample = customers_df.select("city").distinct().first()[0]
city_df = customers_df.filter(col("city") == city_sample)
print(f"Customers in '{city_sample}': {city_df.count()} rows")

# withColumn — add new columns
enriched_df = (
    customers_df
    .withColumn("full_name", col("first_name") + " " + col("last_name"))
    .withColumn("ingestion_ts", current_timestamp())
)
print("After withColumn — new columns added:")
enriched_df.select("customer_id", "full_name", "ingestion_ts").show(3)

## Section 5 — Writing to Delta (Demo — a Preview of Bronze)

After reading and transforming a DataFrame, a real pipeline writes it to **Bronze** as a **Delta table** — a governed Unity Catalog table (`<your-catalog>.bronze.<table>`), built properly starting Day 4.

For today, we practice the write/read mechanics against a scratch demo path instead — same `.write.format("delta")` API, just not the real destination yet.

### Two write modes

| Mode | Behaviour | When to use |
|------|-----------|-------------|
| `overwrite` | Deletes existing data, writes fresh | Small reference tables — safe to fully refresh |
| `append` | Adds new rows, never deletes | Growing tables — add only what is new |

### Path structure for today's demo

```
raw-data/_delta_demo/payment_methods/
raw-data/_delta_demo/customers/
raw-data/_delta_demo/products/
```

After writing, a `_delta_log/` folder appears alongside the Parquet files — that is Delta Lake's transaction log (covered in depth tomorrow in ILT2).

In [ ]:
# ─── Write customers to the demo Delta path (overwrite) ───────────────────────

enriched_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{delta_demo_path}/customers")

# Read it back to verify
demo_customers = spark.read.format("delta").load(f"{delta_demo_path}/customers")
print(f"Demo customers row count: {demo_customers.count()}")
print("Sample of what landed:")
demo_customers.select("customer_id", "full_name", "ingestion_ts").show(5)

In [ ]:
# ─── Write a few more ADLS reference tables to the demo Delta path ────────────

reference_tables = ["payment_methods", "products", "addresses"]

for table_name in reference_tables:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{raw_path}/{table_name}/{table_name}.csv")
    )
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{delta_demo_path}/{table_name}")
    print(f"Written: _delta_demo/{table_name}  ({df.count()} rows)")

print("\nAll 3 reference tables written to the demo Delta path!")

In [ ]:
# ─── Read payments.csv — explore before writing ───────────────────────────────
# Note: orders/order_items are NOT read here — they come from Postgres via
# Lakeflow Connect (Day 2), not an ADLS CSV file. payments is a real
# ADLS-sourced table, so it's a faithful stand-in for this append demo.

payments_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{raw_path}/payments/payments.csv")
)

print(f"Total payments in CSV: {payments_df.count()}")
payments_df.printSchema()
payments_df.show(3, truncate=False)

In [ ]:
# ─── Write payments to the demo path + demo append mode ───────────────────────
from pyspark.sql.functions import current_timestamp

payments_enriched = payments_df.withColumn("_ingestion_ts", current_timestamp())

# Initial write — overwrite to initialise the demo table
payments_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{delta_demo_path}/payments")

count_v1 = spark.read.format("delta").load(f"{delta_demo_path}/payments").count()
print(f"Demo payments after initial write:  {count_v1} rows")

# Simulate new data arriving — append 5 rows (demo only)
new_rows = payments_df.limit(5).withColumn("_ingestion_ts", current_timestamp())
new_rows.write \
    .format("delta") \
    .mode("append") \
    .save(f"{delta_demo_path}/payments")

count_v2 = spark.read.format("delta").load(f"{delta_demo_path}/payments").count()
print(f"Demo payments after append:         {count_v2} rows")
print(f"Rows added by append:               {count_v2 - count_v1}")
print()
print("overwrite = replace everything | append = add without touching existing rows")

## Section 6 — Spark SQL

Once data is written as a Delta table (or registered as a temp view), you can query it with **standard SQL** directly in Databricks — the same SQL you already know.

### Two ways to run SQL

**Option A — `%sql` cell** (pure SQL, no Python):
```sql
%sql
SELECT city, COUNT(*) AS total
FROM bronze_customers_view
GROUP BY city
ORDER BY total DESC
LIMIT 10
```

**Option B — `spark.sql()` inside Python** (returns a DataFrame):
```python
result_df = spark.sql("SELECT city, COUNT(*) AS total FROM bronze_customers_view GROUP BY city")
result_df.show()
```

Use `%sql` for quick exploration. Use `spark.sql()` when you need to pass the result into more PySpark code.

### Register a temp view

Before querying with `%sql`, register your DataFrame as a temporary view:
```python
df.createOrReplaceTempView("my_view_name")
```
Temp views live for the session. For permanent tables, write to a path and use Unity Catalog.

## Spark SQL — Register a Temp View + Query with %sql

To query a DataFrame with SQL, first register it as a **temporary view**:

```python
df.createOrReplaceTempView("view_name")
```

Then in a `%sql` cell:
```sql
SELECT * FROM view_name WHERE city = 'Mumbai' LIMIT 10
```

Or from Python using `spark.sql()`:
```python
result = spark.sql("SELECT city, COUNT(*) FROM view_name GROUP BY city")
result.show()
```

### Practical difference

| | `%sql` | `spark.sql()` |
|-|--------|---------------|
| Returns | Table rendered in UI | PySpark DataFrame |
| Chain transformations? | No | Yes |
| Use case | Quick exploration | Feed result into more Python code |

In [ ]:
# ─── Spark SQL via spark.sql() ────────────────────────────────────────────────

# Register the demo customers table as a temp view
demo_customers = spark.read.format("delta").load(f"{delta_demo_path}/customers")
demo_customers.createOrReplaceTempView("demo_customers_view")

# Query 1: count customers by city
city_counts = spark.sql("""
    SELECT city, COUNT(*) AS customer_count
    FROM demo_customers_view
    GROUP BY city
    ORDER BY customer_count DESC
    LIMIT 10
""")
print("Top 10 cities by customer count:")
city_counts.show()

# The result is a DataFrame — you can chain more transformations
top_city = city_counts.first()
print(f"Top city: {top_city['city']} — {top_city['customer_count']} customers")

## Section 7 — PySpark Aggregations &amp; Joins

### groupBy + agg

```python
from pyspark.sql.functions import count, sum, avg

df.groupBy("status") \
  .agg(
      count("*").alias("total_orders"),
      avg("amount").alias("avg_amount")
  ) \
  .orderBy("total_orders", ascending=False) \
  .show()
```

### join

```python
# Join orders with customers
result = orders_df.join(customers_df, on="customer_id", how="left")
```

Join types: `inner`, `left`, `right`, `full` — same semantics as SQL.

### Common PySpark functions to import

```python
from pyspark.sql.functions import (
    col, lit, when,             # column operations
    count, sum, avg, max, min,  # aggregations
    upper, lower, trim, length, # string functions
    year, month, dayofmonth,    # date functions
    current_timestamp, to_date, # timestamps
    coalesce, isNull, isNotNull # null handling
)
```

> These are all from `pyspark.sql.functions` — import them at the top of your notebook.

## PySpark vs Spark SQL — When to Use Which

Both produce the same result. Choose based on readability.

| Scenario | Recommended |
|----------|-------------|
| Exploration — quick look at data | Spark SQL (`%sql`) |
| Complex multi-step transformations | PySpark (chain methods) |
| Joining 5+ tables | Spark SQL (clearer) |
| Adding computed columns + writing | PySpark (`.withColumn().write`) |
| Team has strong SQL background | Spark SQL |
| Team has strong Python background | PySpark |

### Same query, two ways

```python
# PySpark
result = (
    orders_df
    .groupBy("status")
    .agg(count("*").alias("total"))
    .orderBy("total", ascending=False)
)

# Spark SQL (equivalent)
result = spark.sql("""
    SELECT status, COUNT(*) AS total
    FROM orders_view
    GROUP BY status
    ORDER BY total DESC
""")

In [ ]:
# ─── PySpark aggregation + join demo ─────────────────────────────────────────
from pyspark.sql.functions import count, col

products_df  = spark.read.format("delta").load(f"{delta_demo_path}/products")
addresses_df = spark.read.format("delta").load(f"{delta_demo_path}/addresses")
customers_df = spark.read.format("delta").load(f"{delta_demo_path}/customers")

# Aggregation: products per category
print("Products by category:")
products_df.groupBy("category") \
    .agg(count("*").alias("total_products")) \
    .orderBy(col("total_products").desc()) \
    .show()

# Register both as views and run a SQL join
customers_df.createOrReplaceTempView("demo_customers")
addresses_df.createOrReplaceTempView("demo_addresses")

# SQL join — addresses enriched with the owning customer's name
joined = spark.sql("""
    SELECT a.address_id, a.city, a.state, c.first_name, c.last_name
    FROM demo_addresses a
    LEFT JOIN demo_customers c ON a.customer_id = c.customer_id
    LIMIT 10
""")
print("Addresses joined with customer names:")
joined.show(truncate=False)

## Recap — What We Covered Today

| Topic | Key Takeaway |
|-------|--------------|
| **Databricks** | Cloud platform on Apache Spark — notebooks + cluster + Delta Lake + Jobs |
| **Cluster** | Driver (coordinates) + Workers (process data in parallel) — your code scales automatically |
| **PySpark DataFrame** | Distributed table — same shape as pandas, runs on a cluster |
| **Lazy evaluation** | Transformations build a plan; actions (show, count, write) trigger execution |
| **Read CSV** | `spark.read.option("header","true").option("inferSchema","true").csv(path)` |
| **Transformations** | `select`, `filter`, `withColumn`, `groupBy`, `join` — chain them freely |
| **Write Delta** | `.write.format("delta").mode("overwrite"/"append").save(path)` |
| **Spark SQL** | `df.createOrReplaceTempView("v")` then `spark.sql("SELECT ...")` or `%sql` |

---

## Hands-on (3:00 PM) — What You Will Do

1. Connect to ADLS using the storage key in the Setup cell
2. Read `customers.csv` — explore with `printSchema()`, `show()`, `count()`
3. Apply 3 transformations: `select`, `filter`, `withColumn`
4. Write to a demo Delta path as practice — verify row count
5. Write 2 more reference tables to the demo Delta path
6. Register a temp view and query it with `spark.sql()`

> Use the code cells in Sections 4–6 directly — they are your template. Remember: today's writes go to a scratch practice path, not GlobalMart's real Bronze — that's a proper Unity Catalog table, built starting Day 4.

---

## Coming Up in the Bootcamp

| Day | Topic |
|-----|-------|
| **Day 2** | Lakeflow Connect + CDC — query/cursor-based incremental capture from Postgres into Bronze |
| **Day 3** | Autoloader + Schema Evolution — incremental file ingestion from ADLS |
| **Day 4** | Bronze Layer build — all sources, audit columns, partitioning strategy |
| **Day 6+** | Bronze → Silver transformation, SCD, dimensional modelling |